<a href="https://colab.research.google.com/github/adityaaverma07/Subscription-Conversion-Prediction-/blob/main/Subscription_Conversion_Prediction_ML_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Subscription Conversion Prediction — End-to-End ML Project

## Business Problem
A subscription-based app has many free users, but only a portion of them convert to paid subscriptions. The business wants to **predict which free users are likely to become paid customers** using their engagement behavior.

This can help the business:
- Identify high-potential free users
- Prioritize conversion campaigns
- Personalize offers and messaging
- Improve marketing efficiency
- Understand which engagement signals are associated with conversion

## Machine Learning Problem
This is a **supervised binary classification** problem.

### Target Variable
`Converted`

- `1` → User became a paid subscriber
- `0` → User did not become a paid subscriber

### Input Features
- Age
- Days since signup
- Sessions
- Visits
- Features used
- Average session duration
- Trial days used
- Support interactions

## Models Used
We will train and compare three classification algorithms:

1. **Logistic Regression** — interpretable linear baseline
2. **Decision Tree Classifier** — captures non-linear decision rules
3. **Random Forest Classifier** — ensemble of multiple decision trees

## Evaluation Metrics
Models will be compared using:
- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Confusion Matrix

## Final Architecture

```text
Raw User Data
      ↓
Data Cleaning
      ↓
EDA
      ↓
Feature Engineering
      ↓
Train / Test Split
      ↓
 ┌────────────────────────────┐
 │ Logistic Regression        │
 │ Decision Tree              │
 │ Random Forest              │
 └────────────────────────────┘
      ↓
Model Evaluation
      ↓
Accuracy | Precision | Recall
F1 | ROC-AUC | Confusion Matrix
      ↓
Select / Tune Model
      ↓
Predict Conversion Probability
```


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    RocCurveDisplay
)

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)


## 2. Load Dataset

Upload `subscription_conversion_100_users.csv` to Colab, then run the next cell.

In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))

df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
display(df.head())


## 3. Understand the Dataset

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing_Values"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
display(df["Converted"].value_counts().rename({0: "Not Converted", 1: "Converted"}))


## 4. Data Cleaning

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates().copy()

# Handle missing numerical values with median
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print("Shape after cleaning:", df.shape)
print("Total missing values:", df.isnull().sum().sum())


## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution
plt.figure(figsize=(6, 4))
df["Converted"].value_counts().sort_index().plot(kind="bar")
plt.xticks([0, 1], ["Not Converted", "Converted"], rotation=0)
plt.ylabel("Number of Users")
plt.title("Subscription Conversion Distribution")
plt.show()


In [ ]:
# Compare average engagement metrics by conversion
eda_cols = [
    "Sessions",
    "Visits",
    "Features_Used",
    "Avg_Session_Minutes",
    "Trial_Days_Used",
    "Support_Interactions"
]

display(df.groupby("Converted")[eda_cols].mean().round(2))


In [ ]:
# Feature distributions by target
for col in eda_cols:
    plt.figure(figsize=(7, 4))
    df.boxplot(column=col, by="Converted")
    plt.title(f"{col} vs Conversion")
    plt.suptitle("")
    plt.xlabel("Converted (0 = No, 1 = Yes)")
    plt.ylabel(col)
    plt.show()


## 6. Feature Engineering

Create additional engagement features that may provide useful signals to the models.

Examples:
- Sessions per day since signup
- Visits per session
- Features used per session
- Engagement score


In [ ]:
# Avoid division by zero
df["Sessions_Per_Day"] = df["Sessions"] / df["Days_Since_Signup"].clip(lower=1)
df["Visits_Per_Session"] = df["Visits"] / df["Sessions"].clip(lower=1)
df["Features_Per_Session"] = df["Features_Used"] / df["Sessions"].clip(lower=1)

# A simple composite engagement score
df["Engagement_Score"] = (
    df["Sessions"] +
    df["Visits"] +
    (df["Features_Used"] * 2) +
    df["Trial_Days_Used"]
)

display(df.head())


## 7. Define Features and Target

**Target variable:** `Converted`

`User_ID` is excluded because it is only an identifier and has no predictive meaning.


In [ ]:
target = "Converted"

feature_cols = [
    "Age",
    "Days_Since_Signup",
    "Sessions",
    "Visits",
    "Features_Used",
    "Avg_Session_Minutes",
    "Trial_Days_Used",
    "Support_Interactions",
    "Sessions_Per_Day",
    "Visits_Per_Session",
    "Features_Per_Session",
    "Engagement_Score"
]

X = df[feature_cols]
y = df[target]

print("Number of features:", X.shape[1])
print("Target distribution:")
print(y.value_counts())


## 8. Train / Test Split

We use an **80/20 stratified split**.

Stratification helps maintain a similar proportion of converted and non-converted users in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 9. Model 1 — Logistic Regression

Logistic Regression provides an interpretable baseline for binary classification.

Standardization is included in a pipeline because Logistic Regression generally benefits from features being on comparable scales.


In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]


## 10. Model 2 — Decision Tree

A Decision Tree learns rule-based splits and can capture non-linear relationships.


In [ ]:
decision_tree_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=5,
    random_state=42
)

decision_tree_model.fit(X_train, y_train)

tree_pred = decision_tree_model.predict(X_test)
tree_prob = decision_tree_model.predict_proba(X_test)[:, 1]


## 11. Model 3 — Random Forest

Random Forest combines many decision trees to create an ensemble prediction and generally provides a robust baseline for tabular classification.


In [ ]:
random_forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=5,
    random_state=42,
    class_weight="balanced"
)

random_forest_model.fit(X_train, y_train)

rf_pred = random_forest_model.predict(X_test)
rf_prob = random_forest_model.predict_proba(X_test)[:, 1]


## 12. Compare All Three Models

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

results = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, logistic_pred, logistic_prob),
    evaluate_model("Decision Tree", y_test, tree_pred, tree_prob),
    evaluate_model("Random Forest", y_test, rf_pred, rf_prob)
])

display(results.round(4))


## 13. Cross-Validation Comparison

With only 100 users, a single train/test split can produce unstable estimates. We therefore also use 5-fold stratified cross-validation for a more robust comparison.


In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree_model,
    "Random Forest": random_forest_model
}

cv_results = []

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    cv_results.append({
        "Model": name,
        "Mean ROC-AUC": scores.mean(),
        "Std ROC-AUC": scores.std()
    })

cv_results = pd.DataFrame(cv_results)
display(cv_results.round(4))


## 14. Confusion Matrices

In [ ]:
predictions = {
    "Logistic Regression": logistic_pred,
    "Decision Tree": tree_pred,
    "Random Forest": rf_pred
}

for name, pred in predictions.items():
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f"Confusion Matrix — {name}")
    plt.show()


## 15. Classification Reports

In [ ]:
for name, pred in predictions.items():
    print("=" * 70)
    print(name)
    print("=" * 70)
    print(classification_report(
        y_test,
        pred,
        target_names=["Not Converted", "Converted"],
        zero_division=0
    ))


## 16. ROC Curve Comparison

In [ ]:
plt.figure(figsize=(8, 6))

RocCurveDisplay.from_predictions(
    y_test, logistic_prob, name="Logistic Regression"
)
RocCurveDisplay.from_predictions(
    y_test, tree_prob, name="Decision Tree"
)
RocCurveDisplay.from_predictions(
    y_test, rf_prob, name="Random Forest"
)

plt.title("ROC Curve Comparison")
plt.show()


## 17. Feature Importance — Random Forest

Random Forest provides a useful view of which input features contributed most to its predictions.

**Important:** feature importance shows association with the model's predictions; it does not prove that a feature causes conversion.


In [ ]:
importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": random_forest_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance_df)

plt.figure(figsize=(8, 5))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("Random Forest Feature Importance")
plt.show()


## 18. Hyperparameter Tuning — Random Forest

We tune the Random Forest using cross-validation.

The goal is not simply to maximize one test-set metric, but to find a configuration that generalizes better.


In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 8, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        class_weight="balanced"
    ),
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)
print("\nBest cross-validation ROC-AUC:", round(grid_search.best_score_, 4))


## 19. Evaluate Tuned Random Forest

In [ ]:
tuned_rf = grid_search.best_estimator_

tuned_rf_pred = tuned_rf.predict(X_test)
tuned_rf_prob = tuned_rf.predict_proba(X_test)[:, 1]

tuned_result = pd.DataFrame([
    evaluate_model(
        "Tuned Random Forest",
        y_test,
        tuned_rf_pred,
        tuned_rf_prob
    )
])

display(tuned_result.round(4))

print(classification_report(
    y_test,
    tuned_rf_pred,
    target_names=["Not Converted", "Converted"],
    zero_division=0
))


## 20. Final Model and Conversion Probability

For a real deployment, the final model should be selected based on the business objective and validation evidence.

The model below demonstrates how to produce a **conversion probability** for each test user.


In [ ]:
# Demonstration using the tuned Random Forest
final_model = tuned_rf

conversion_probability = final_model.predict_proba(X_test)[:, 1]

prediction_output = X_test.copy()
prediction_output["Actual_Converted"] = y_test.values
prediction_output["Conversion_Probability"] = conversion_probability
prediction_output["Predicted_Converted"] = (
    conversion_probability >= 0.50
).astype(int)

prediction_output = prediction_output.sort_values(
    "Conversion_Probability",
    ascending=False
)

display(prediction_output.head(10))


## 21. Business Interpretation

The final output can be used to segment free users by predicted conversion probability.

Example:

- High predicted probability → potentially prioritize for conversion messaging
- Medium probability → test personalized offers or onboarding
- Low probability → focus on engagement/onboarding rather than immediate conversion campaigns

These are **business actions to test**, not guarantees of user behavior.

### Important modeling note
This synthetic dataset is intended for learning and portfolio demonstration. With only 100 users, model metrics can vary substantially depending on the train/test split. A production system should use a much larger historical dataset, time-based validation where appropriate, leakage checks, calibration, monitoring, and periodic retraining.


# Final Project Summary

### Business Problem
Predict whether a free user will convert to a paid subscriber.

### Problem Type
Binary Classification

### Target Variable
`Converted`

### Models
1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier

### Evaluation Metrics
- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Confusion Matrix

### Final Output
A probability between 0 and 1 representing the model's estimated likelihood that a free user will convert.

### End-to-End Pipeline

```text
Raw User Data
      ↓
Data Cleaning
      ↓
EDA
      ↓
Feature Engineering
      ↓
Train / Test Split
      ↓
Logistic Regression
Decision Tree
Random Forest
      ↓
Model Comparison
      ↓
Hyperparameter Tuning
      ↓
Final Model
      ↓
Conversion Probability
```
